# Civic Image Verification — AI Detection + Duplicate Detection

This Colab notebook provides one pipeline for:
- image validation
- SHA-256 exact duplicate detection
- pHash / dHash / aHash near-duplicate detection
- CLIP visual similarity
- EXIF metadata inspection
- open-source AI-generated image classification
- civic-issue zero-shot classification
- final verification / review decision

**Important:** AI-image detection is probabilistic. `LIKELY_AI`, `LIKELY_REAL`, and `UNCERTAIN` are safer labels than claiming an image is definitively real or fake.


In [ ]:
!pip -q install torch torchvision transformers accelerate pillow imagehash opencv-python sentence-transformers scikit-learn piexif


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 4.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
from PIL import Image, ExifTags
import imagehash
import torch
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Device: cpu


## 1. Configuration


In [ ]:
ALLOWED_FORMATS = {'JPEG', 'PNG', 'WEBP', 'BMP', 'TIFF'}
MAX_IMAGE_SIZE_MB = 15

PHASH_NEAR_THRESHOLD = 8
DHASH_NEAR_THRESHOLD = 8
AHASH_NEAR_THRESHOLD = 8
VISUAL_SIMILARITY_THRESHOLD = 0.90
HIGH_VISUAL_SIMILARITY = 0.96

CIVIC_ISSUE_CATEGORIES = [
    'pothole', 'garbage or waste', 'broken streetlight',
    'graffiti', 'blocked drain', 'damaged road sign',
    'illegal parking', 'public nuisance', 'roadkill',
    'fallen tree', 'water leak', 'fire hazard'
]


## 2. Image validation


In [ ]:
def validate_image(image_path):
    result = {'valid': False, 'format': None, 'width': None, 'height': None, 'message': ''}
    try:
        if not os.path.exists(image_path):
            result['message'] = 'File does not exist.'
            return result
        size_mb = os.path.getsize(image_path) / (1024 * 1024)
        if size_mb > MAX_IMAGE_SIZE_MB:
            result['message'] = f'Image is too large: {size_mb:.2f} MB'
            return result
        image = Image.open(image_path)
        result['format'] = image.format
        result['width'], result['height'] = image.size
        if image.format not in ALLOWED_FORMATS:
            result['message'] = f'Unsupported format: {image.format}'
            return result
        image.verify()
        result['valid'] = True
        result['message'] = 'Valid image.'
    except Exception as e:
        result['message'] = f'Invalid image: {e}'
    return result


## 3. SHA-256 exact duplicate detection


In [ ]:
def calculate_sha256(image_path):
    sha256 = hashlib.sha256()
    with open(image_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            sha256.update(chunk)
    return sha256.hexdigest()


## 4. Perceptual hashes


In [ ]:
def calculate_perceptual_hashes(image_path):
    image = Image.open(image_path).convert('RGB')
    return {
        'phash': str(imagehash.phash(image)),
        'dhash': str(imagehash.dhash(image)),
        'ahash': str(imagehash.average_hash(image)),
        'whash': str(imagehash.whash(image))
    }


## 5. CLIP visual embeddings


In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading CLIP embedding model...')
embedding_model = SentenceTransformer('clip-ViT-B-32', device=DEVICE)
print('Loaded.')

def get_image_embedding(image_path):
    image = Image.open(image_path).convert('RGB')
    return embedding_model.encode(image, normalize_embeddings=True)


Loading CLIP embedding model...


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loaded.


## 6. Compare two images


In [ ]:
def compare_images(image1_path, image2_path):
    sha1, sha2 = calculate_sha256(image1_path), calculate_sha256(image2_path)
    if sha1 == sha2:
        return {
            'match_type': 'EXACT_DUPLICATE', 'exact_duplicate': True,
            'near_duplicate': True, 'visual_similarity': 1.0,
            'phash_distance': 0, 'dhash_distance': 0, 'ahash_distance': 0
        }

    h1, h2 = calculate_perceptual_hashes(image1_path), calculate_perceptual_hashes(image2_path)
    phash_distance = imagehash.hex_to_hash(h1['phash']) - imagehash.hex_to_hash(h2['phash'])
    dhash_distance = imagehash.hex_to_hash(h1['dhash']) - imagehash.hex_to_hash(h2['dhash'])
    ahash_distance = imagehash.hex_to_hash(h1['ahash']) - imagehash.hex_to_hash(h2['ahash'])

    emb1, emb2 = get_image_embedding(image1_path), get_image_embedding(image2_path)
    similarity = float(cosine_similarity([emb1], [emb2])[0][0])

    hash_matches = sum([
        phash_distance <= PHASH_NEAR_THRESHOLD,
        dhash_distance <= DHASH_NEAR_THRESHOLD,
        ahash_distance <= AHASH_NEAR_THRESHOLD
    ])

    if hash_matches >= 2 and similarity >= HIGH_VISUAL_SIMILARITY:
        match_type = 'NEAR_DUPLICATE'
    elif similarity >= VISUAL_SIMILARITY_THRESHOLD:
        match_type = 'VISUALLY_SIMILAR'
    else:
        match_type = 'DIFFERENT'

    return {
        'match_type': match_type,
        'exact_duplicate': False,
        'near_duplicate': match_type == 'NEAR_DUPLICATE',
        'visual_similarity': round(similarity, 4),
        'phash_distance': phash_distance,
        'dhash_distance': dhash_distance,
        'ahash_distance': ahash_distance
    }


## 7. EXIF metadata analysis


In [ ]:
def extract_exif(image_path):
    result = {}
    try:
        image = Image.open(image_path)
        exif = image.getexif()
        if not exif:
            return result
        for tag_id, value in exif.items():
            tag = ExifTags.TAGS.get(tag_id, tag_id)
            result[str(tag)] = str(value)
    except Exception as e:
        result['error'] = str(e)
    return result

def metadata_analysis(image_path):
    exif = extract_exif(image_path)
    if not exif:
        return {'metadata_present': False, 'message': 'No EXIF metadata found.'}
    important = ['Make', 'Model', 'DateTimeOriginal', 'GPSInfo', 'Software']
    return {
        'metadata_present': True,
        'available_fields': [x for x in important if x in exif],
        'metadata': exif
    }


## 8. Open-source AI-generated image detector

The model ID below is intentionally configurable. Before deployment, verify its current Hugging Face model card and labels. Never assume arbitrary labels mean `AI` or `REAL`.


In [ ]:
from transformers import pipeline

# Replace this with a currently available open-source AI-image detector
# whose model card confirms the labels and intended use.
AI_DETECTOR_MODEL = 'umm-maybe/AI-image-detector'

print('Loading AI detector:', AI_DETECTOR_MODEL)
ai_detector = pipeline(
    'image-classification',
    model=AI_DETECTOR_MODEL,
    device=0 if torch.cuda.is_available() else -1
)
print('Model labels:', getattr(ai_detector.model.config, 'id2label', {}))


Loading AI detector: umm-maybe/AI-image-detector


config.json: reconstructing file:   0%|          |  0.00B /   937B            

config.json: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  348MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  347MB            

model.safetensors: downloading bytes:           |  0.00B            

preprocessor_config.json: reconstructing file:   0%|          |  0.00B /   240B            

preprocessor_config.json: downloading bytes:           |  0.00B            

Model labels: {0: 'artificial', 1: 'human'}


In [ ]:
AI_LABELS = {'ai', 'artificial', 'generated', 'fake', 'synthetic'}
REAL_LABELS = {'real', 'human', 'authentic', 'natural'}

def detect_ai_image(image_path):
    image = Image.open(image_path).convert('RGB')
    results = sorted(ai_detector(image), key=lambda x: x['score'], reverse=True)
    return {
        'predictions': [
            {'label': r['label'], 'score': round(float(r['score']), 4)}
            for r in results
        ]
    }

def classify_ai_probability(image_path):
    predictions = detect_ai_image(image_path)['predictions']
    ai_score = sum(x['score'] for x in predictions if any(w in x['label'].lower() for w in AI_LABELS))
    real_score = sum(x['score'] for x in predictions if any(w in x['label'].lower() for w in REAL_LABELS))
    total = ai_score + real_score
    ai_probability = ai_score / total if total > 0 else None
    if ai_probability is None:
        status = 'UNCERTAIN'
    elif ai_probability >= 0.80:
        status = 'LIKELY_AI'
    elif ai_probability <= 0.20:
        status = 'LIKELY_REAL'
    else:
        status = 'UNCERTAIN'
    return {
        'ai_probability': round(ai_probability, 4) if ai_probability is not None else None,
        'status': status,
        'raw_predictions': predictions
    }


## 9. Civic issue classification with open CLIP


In [ ]:
from transformers import CLIPProcessor, CLIPModel

CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

def classify_civic_issue(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = clip_processor(
        text=CIVIC_ISSUE_CATEGORIES,
        images=image,
        return_tensors='pt',
        padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probabilities = outputs.logits_per_image.softmax(dim=1)[0]
    results = [
        {'category': CIVIC_ISSUE_CATEGORIES[i], 'confidence': float(probabilities[i])}
        for i in range(len(CIVIC_ISSUE_CATEGORIES))
    ]
    results.sort(key=lambda x: x['confidence'], reverse=True)
    return {
        'category': results[0]['category'],
        'confidence': round(results[0]['confidence'], 4),
        'all_predictions': results
    }


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## 10. Full single-image analysis


In [ ]:
def analyze_image(image_path):
    validation = validate_image(image_path)
    if not validation['valid']:
        return {'success': False, 'error': validation['message']}

    return {
        'success': True,
        'image': {
            'path': image_path,
            'format': validation['format'],
            'width': validation['width'],
            'height': validation['height']
        },
        'hashes': {
            'sha256': calculate_sha256(image_path),
            **calculate_perceptual_hashes(image_path)
        },
        'metadata': metadata_analysis(image_path),
        'ai_detection': classify_ai_probability(image_path),
        'civic_issue': classify_civic_issue(image_path)
    }


## 11. Image database and duplicate search


In [ ]:
image_database = []

def add_image_to_database(image_path, image_id):
    record = {
        'image_id': image_id,
        'path': image_path,
        'sha256': calculate_sha256(image_path),
        'hashes': calculate_perceptual_hashes(image_path),
        'embedding': get_image_embedding(image_path)
    }
    image_database.append(record)
    return record

def search_duplicates(image_path):
    query_sha = calculate_sha256(image_path)
    query_hashes = calculate_perceptual_hashes(image_path)
    query_embedding = get_image_embedding(image_path)
    matches = []

    for record in image_database:
        if query_sha == record['sha256']:
            matches.append({'image_id': record['image_id'], 'match_type': 'EXACT_DUPLICATE', 'similarity': 1.0})
            continue

        phash_distance = imagehash.hex_to_hash(query_hashes['phash']) - imagehash.hex_to_hash(record['hashes']['phash'])
        dhash_distance = imagehash.hex_to_hash(query_hashes['dhash']) - imagehash.hex_to_hash(record['hashes']['dhash'])
        ahash_distance = imagehash.hex_to_hash(query_hashes['ahash']) - imagehash.hex_to_hash(record['hashes']['ahash'])
        similarity = float(cosine_similarity([query_embedding], [record['embedding']])[0][0])

        hash_matches = sum([
            phash_distance <= PHASH_NEAR_THRESHOLD,
            dhash_distance <= DHASH_NEAR_THRESHOLD,
            ahash_distance <= AHASH_NEAR_THRESHOLD
        ])

        if hash_matches >= 2 and similarity >= HIGH_VISUAL_SIMILARITY:
            match_type = 'NEAR_DUPLICATE'
        elif similarity >= VISUAL_SIMILARITY_THRESHOLD:
            match_type = 'VISUALLY_SIMILAR'
        else:
            continue

        matches.append({
            'image_id': record['image_id'],
            'match_type': match_type,
            'visual_similarity': round(similarity, 4),
            'phash_distance': phash_distance,
            'dhash_distance': dhash_distance,
            'ahash_distance': ahash_distance
        })

    matches.sort(key=lambda x: x.get('visual_similarity', 0), reverse=True)
    return matches


## 12. Final verification decision


In [ ]:
def final_verification(ai_detection, duplicate_results, civic_confidence):
    flags = []
    if ai_detection['status'] == 'LIKELY_AI':
        flags.append('AI_GENERATED_IMAGE')

    for match in duplicate_results:
        if match['match_type'] == 'EXACT_DUPLICATE':
            flags.append('EXACT_DUPLICATE')
            break
        if match['match_type'] == 'NEAR_DUPLICATE':
            flags.append('NEAR_DUPLICATE')

    if civic_confidence < 0.45:
        flags.append('LOW_CIVIC_CLASSIFICATION_CONFIDENCE')

    if 'EXACT_DUPLICATE' in flags:
        status = 'REJECT_DUPLICATE'
    elif 'AI_GENERATED_IMAGE' in flags:
        status = 'REVIEW_AI_IMAGE'
    elif 'NEAR_DUPLICATE' in flags:
        status = 'REVIEW_DUPLICATE'
    elif 'LOW_CIVIC_CLASSIFICATION_CONFIDENCE' in flags:
        status = 'MANUAL_REVIEW'
    else:
        status = 'LIKELY_VALID'

    return {
        'status': status,
        'flags': flags,
        'requires_manual_review': status != 'LIKELY_VALID'
    }


## 13. Upload and run


In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded.keys()))
print('Selected:', image_path)

result = analyze_image(image_path)
print(json.dumps(result, indent=2, default=str))


Saving Screenshot 2026-08-19 at 5.27.09 PM.png to Screenshot 2026-08-19 at 5.27.09 PM.png
Selected: Screenshot 2026-08-19 at 5.27.09 PM.png
{
  "success": true,
  "image": {
    "path": "Screenshot 2026-08-19 at 5.27.09\u202fPM.png",
    "format": "PNG",
    "width": 2090,
    "height": 798
  },
  "hashes": {
    "sha256": "c51ce91c8f80c677062bec30cd0c107cc03f0986456791b6640ced1ee153418f",
    "phash": "a4992d669b8c7d23",
    "dhash": "fefefefeccc6c6a6",
    "ahash": "3f1f1f3f67020300",
    "whash": "3f3f1f3f67030300"
  },
  "metadata": {
    "metadata_present": true,
    "available_fields": [],
    "metadata": {
      "ResolutionUnit": "2",
      "ExifOffset": "78",
      "XResolution": "144.0",
      "YResolution": "144.0"
    }
  },
  "ai_detection": {
    "ai_probability": 0.0553,
    "status": "LIKELY_REAL",
    "raw_predictions": [
      {
        "label": "human",
        "score": 0.9447
      },
      {
        "label": "artificial",
        "score": 0.0553
      }
    ]
  },
 

## 14. Compare a second image


In [ ]:
uploaded2 = files.upload()
image2_path = next(iter(uploaded2.keys()))

comparison = compare_images(image_path, image2_path)
print(json.dumps(comparison, indent=2))


Saving Screenshot 2026-08-19 at 5.27.33 PM.png to Screenshot 2026-08-19 at 5.27.33 PM.png
{
  "match_type": "DIFFERENT",
  "exact_duplicate": false,
  "near_duplicate": false,
  "visual_similarity": 0.876,
  "phash_distance": 38,
  "dhash_distance": 40,
  "ahash_distance": 34
}


## 15. Add images to the duplicate database


In [ ]:
add_image_to_database(image_path, 'IMG-001')
add_image_to_database(image2_path, 'IMG-002')

matches = search_duplicates(image_path)
print(json.dumps(matches, indent=2))


[
  {
    "image_id": "IMG-001",
    "match_type": "EXACT_DUPLICATE",
    "similarity": 1.0
  }
]
